# Chapter 2: Content-Based & Hybrid Recommender

This notebook extends the collaborative filtering recommender with:
1. **Content-based recommendations** using TF-IDF on genres and titles
2. **Hybrid retrieval** combining behavioral and content signals
3. **Three-signal ranking** (content similarity + behavioral similarity + popularity)

All approaches use the same four-stage framework. At the end we show how the content-based retrieval plugs into the `recsys` framework.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yourusername/modern-recommender-systems/blob/main/notebooks/chapter-02/content-based_fourstage_recommender_example.ipynb)


In [ ]:
# Colab setup: clone the repo and install the `recsys` package.
# This is a no-op when running locally.
import sys

if "google.colab" in sys.modules:
    import os
    from pathlib import Path

    REPO_URL = "https://github.com/yourusername/modern-recommender-systems.git"
    BRANCH = "main"
    REPO_DIR = Path("/content/modern-recommender-systems")

    if not REPO_DIR.exists():
        os.system(f"git clone -q -b {BRANCH} {REPO_URL} {REPO_DIR}")
    os.system(f"pip install -q -e {REPO_DIR}")

    # Make the notebook's working directory match the local layout
    # (notebooks/chapter-02/) so relative data paths resolve.
    nb_dir = REPO_DIR / "notebooks" / "chapter-02"
    os.chdir(nb_dir)
    print(f"✓ Colab environment ready (cwd={nb_dir})")


## Setup

In [ ]:
import numpy as np
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

from sklearn.metrics.pairwise import cosine_similarity

from recsys.data.loaders import load_movielens
from recsys.utils.colab import get_data_path

In [ ]:
DATA_PATH = get_data_path()
ratings, movies = load_movielens("ml-25m", data_dir=DATA_PATH)
print(ratings.head())
print(f"ratings shape: {ratings.shape}")
print(movies.head())
print(f"movies shape: {movies.shape}")

### Rebuild collaborative filtering components

We need the user-item matrix, item similarity, popularity scores, and helper functions from the first notebook.

In [ ]:
from scipy.sparse import csr_matrix

user_ids = ratings['userId'].unique()
movie_ids = ratings['movieId'].unique()
user_to_idx = {uid: idx for idx, uid in enumerate(user_ids)}
movie_to_idx = {mid: idx for idx, mid in enumerate(movie_ids)}
idx_to_movie = {idx: mid for mid, idx in movie_to_idx.items()}

rows = [user_to_idx[uid] for uid in ratings['userId']]
cols = [movie_to_idx[mid] for mid in ratings['movieId']]
data = [1] * len(ratings)

user_item_matrix = csr_matrix(
    (data, (rows, cols)),
    shape=(len(user_ids), len(movie_ids))
)

print(f"Computing item-item similarity (this may take a few minutes)...")
item_similarity = cosine_similarity(user_item_matrix.T, dense_output=False)
print(f"Done. Shape: {item_similarity.shape}")

In [ ]:
# Helper functions from the first notebook
movie_counts = ratings['movieId'].value_counts()
max_count = movie_counts.max()
popularity_scores = {mid: float(count / max_count) for mid, count in movie_counts.items()}

def get_user_history(user_id):
    return set(mid for mid in ratings[ratings['userId'] == user_id]['movieId'].values)

def filter_watched(candidates, user_id):
    user_history = get_user_history(user_id)
    return [item for item in candidates if item['movie_id'] not in user_history]

def add_popularity_scores(candidates):
    for item in candidates:
        item['popularity'] = popularity_scores.get(item['movie_id'], 0.0)
    return candidates

def retrieve_similar_items(movie_id, k=100):
    if movie_id not in movie_to_idx: return []
    movie_idx = movie_to_idx[movie_id]
    similarities = item_similarity[movie_idx].toarray()[0]
    top_indices = np.argsort(similarities)[-(k+1):-1][::-1]
    return [{'movie_id': idx_to_movie[idx], 'similarity': float(similarities[idx])} for idx in top_indices]

## Content-Based Recommendations

Content-based recommendations use item metadata — genres, descriptions, actors — to find similar items. They work immediately for new items, without behavioral data.

### Sample metadata

In [ ]:
print("Sample movies:\n")
for _, row in movies.head(5).iterrows():
    print(f"  {row['title']}  Genres: {row['genres']}")

### Creating content vectors with TF-IDF

In [ ]:
# Listing 2.17: Creating TF-IDF vectors
from sklearn.feature_extraction.text import TfidfVectorizer

movies["clean_title"] = (
    movies["title"]
    .str.replace(r"\s*\((?:19|20)\d{2}(?:-(?:19|20)\d{2})?\)\s*$", "", regex=True)
    .str.strip()
) #A
movies['content'] = movies['clean_title'] + ' ' + movies['genres'].fillna('') #B

vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words='english', #C
    token_pattern=r'(?u)\b\w+\b'
)
content_vectors = vectorizer.fit_transform(movies['content']) #D

#A Remove years from titles
#B Combine title and genres for richer representation
#C Remove common English words
#D Create TF-IDF vectors, limit to 500 features

print(f"Content vectors shape: {content_vectors.shape}, Memory: {content_vectors.data.nbytes / 1e6:.1f} MB")

In [ ]:
# Inspect top features
feature_names = vectorizer.get_feature_names_out()
mean_tfidf = content_vectors.mean(axis=0).A1
top_features = np.argsort(mean_tfidf)[-20:][::-1]
print("Top 20 TF-IDF features (by mean score):")
for idx in top_features:
    print(f"  {feature_names[idx]:<20} {mean_tfidf[idx]:.4f}")

### Computing content similarity

In [ ]:
# Listing 2.18: Content-based retrieval function
movie_id_to_idx = {mid: idx for idx, mid in enumerate(movies['movieId'])} #A
idx_to_movie_id = {idx: mid for mid, idx in movie_id_to_idx.items()} #A

def retrieve_similar_by_content(movie_id, k=100):
    if movie_id not in movie_id_to_idx: return []
    movie_idx = movie_id_to_idx[movie_id] #B
    movie_vector = content_vectors[movie_idx] #B
    similarities = cosine_similarity(movie_vector, content_vectors)[0] #C
    top_indices = np.argsort(similarities)[-(k+1):-1][::-1] #D
    return [{'movie_id': idx_to_movie_id[idx],
             'content_similarity': float(similarities[idx])} for idx in top_indices] #E

#A Create movie ID to index mapping
#B Get vector for seed movie
#C Compute similarity to all other movies
#D Find top k most similar
#E Return candidates with scores

In [ ]:
# Test on the cold-start movie
new_movie_id = '209157'
new_movie = movies[movies['movieId'] == new_movie_id]
print(f"ID: {new_movie_id}, Title: {new_movie['title'].values[0]}")
print(f"Genres: {new_movie['genres'].values[0]}")
print(f"Ratings: {len(ratings[ratings['movieId'] == new_movie_id])}")

content_candidates = retrieve_similar_by_content(new_movie_id, k=10)
print(f"\nSimilar movies based on content:")
for item in content_candidates[:3]:
    mid = item['movie_id']
    title = movies[movies['movieId'] == mid]['title'].values[0]
    genres = movies[movies['movieId'] == mid]['genres'].values[0]
    print(f"  {mid}. {title}, sim: {item['content_similarity']:.4f}, genres: {genres}")

### Fitting Into the Four-Stage Framework

In [ ]:
# Listing 2.19: Content-based recommender using the framework
def recommend_content_based(user_id, k=10, content_weight=0.7):
    user_history = get_user_history(user_id)
    
    if len(user_history) == 0:
        return [{'movie_id': mid} for mid in movie_counts.head(k).index.tolist()]
    all_candidates = {}
    
    for movie_id in list(user_history)[-20:]:
        candidates = retrieve_similar_by_content(movie_id, k=50) #A
            
        for item in candidates:
            mid = item['movie_id']
            score = item['content_similarity']
            if mid in all_candidates:
                all_candidates[mid] = max(all_candidates[mid], score)
            else:
                all_candidates[mid] = score
                
    candidates = [{'movie_id': mid, 'content_similarity': score}
                  for mid, score in all_candidates.items()]
    candidates = filter_watched(candidates, user_id) #B
    candidates = add_popularity_scores(candidates)   #C
    for item in candidates: #D
        item['final_score'] = content_weight * item['content_similarity'] + (1 - content_weight) * item['popularity']
    return sorted(candidates, key=lambda x: x['final_score'], reverse=True)[:k]

#A Only this line changed — content retrieval instead of collaborative
#B Stage 2: Filter  #C Stage 3: Score  #D Stage 4: Rank

In [ ]:
# Compare collaborative vs content-based for user 1
def recommend_collab(user_id, k=10, similarity_weight=0.7):
    user_history = get_user_history(user_id)
    
    if len(user_history) == 0:
        return [{'movie_id': int(mid)} for mid in movie_counts.head(k).index.tolist()]
    all_candidates = {}
    for movie_id in list(user_history)[-20:]:
        for item in retrieve_similar_items(movie_id, k=50):
            mid = item['movie_id']
            all_candidates[mid] = max(all_candidates.get(mid, 0), item['similarity'])
    candidates = [{'movie_id': mid, 'similarity': score} for mid, score in all_candidates.items()]
    candidates = add_popularity_scores(candidates)
    filtered = filter_watched(candidates, user_id)
    for item in filtered:
        item['final_score'] = similarity_weight * item['similarity'] + (1 - similarity_weight) * item['popularity']
    return sorted(filtered, key=lambda x: x['final_score'], reverse=True)[:k]

collab_recs = recommend_collab('1', k=5)
content_recs = recommend_content_based('1', k=5)
print(content_recs)
print(f"{'#':<3} {'Collaborative Filtering':<42} {'Content-Based':<42}")
print('-' * 87)
for i in range(5):
    ct = movies[movies['movieId'] == collab_recs[i]['movie_id']]['title'].values[0]
    cbt = movies[movies['movieId'] == content_recs[i]['movie_id']]['title'].values[0]
    print(f"{i+1:<3} {ct:<32} {collab_recs[i]['final_score']:.3f}   {cbt:<32} {content_recs[i]['final_score']:.3f}")

## Combining Behavioral and Content Signals

In [ ]:
# Listing 2.20: Hybrid retrieval
def retrieve_hybrid(movie_id, k=100):
    content_candidates = retrieve_similar_by_content(movie_id, k=k//2) #A
    behavioral_candidates = retrieve_similar_items(movie_id, k=k//2) #B
    all_candidates = {}
    for item in content_candidates:
        mid = item['movie_id']
        all_candidates[mid] = {'movie_id': mid, 'content_similarity': item['content_similarity'], 'behavioral_similarity': 0.0}
    for item in behavioral_candidates: #C
        mid = item['movie_id']
        if mid in all_candidates:
            all_candidates[mid]['behavioral_similarity'] = item['similarity']
        else:
            all_candidates[mid] = {'movie_id': mid, 'content_similarity': 0.0, 'behavioral_similarity': item['similarity']}
    return list(all_candidates.values())
#A Get candidates from content similarity
#B Get candidates from behavioral similarity
#C Merge, keeping both scores for each item

In [ ]:
# Listing 2.21: Three-signal ranking
def rank_three_signals(candidates, content_weight=0.3, behavioral_weight=0.5, k=10):
    popularity_weight = 1.0 - content_weight - behavioral_weight #A
    for item in candidates: #B
        item['final_score'] = (
            content_weight * item.get('content_similarity', 0) +
            behavioral_weight * item.get('behavioral_similarity', 0) +
            popularity_weight * item.get('popularity', 0)
        )
    return sorted(candidates, key=lambda x: x['final_score'], reverse=True)[:k]
#A Calculate weights (must sum to 1.0)
#B Weighted combination of all three signals

In [ ]:
# Listing 2.22: Using three-signal ranker with Toy Story as seed
seed_movie_id = '1'
test_user = '1'


candidates = retrieve_hybrid(seed_movie_id, k=100) #A
candidates = filter_watched(candidates, test_user)  #B
candidates = add_popularity_scores(candidates)      #C
ranked = rank_three_signals(candidates, content_weight=0.3, behavioral_weight=0.5) #D

#A Retrieve from both sources  #B Filter  #C Score  #D Rank

seed_title = movies[movies['movieId'] == seed_movie_id]['title'].values[0]
print(f"Hybrid recommendations seeded from '{seed_title}':\n")
print(f"{'#':<3} {'Title':<38} {'Content':>8} {'Behavioral':>10} {'Popularity':>10} {'Final':>8}")
print('-' * 80)
for i, item in enumerate(ranked[:10], 1):
    title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
    print(f"{i:<3} {title:<38} {item.get('content_similarity',0):>8.3f} "
          f"{item.get('behavioral_similarity',0):>10.3f} "
          f"{item.get('popularity',0):>10.3f} {item['final_score']:>8.3f}")

### Full hybrid recommender iterating over user history

In [ ]:
#TODO reuse the weighted hybrid retrieval from chapter one. 
def recommend_hybrid(user_id, k=10, content_weight=0.3, behavioral_weight=0.5):
    user_history = get_user_history(user_id)
    if len(user_history) == 0:
        return [{'movie_id': int(mid)} for mid in movie_counts.head(k).index.tolist()]
    all_candidates = {}
    for movie_id in list(user_history)[-20:]:
        for item in retrieve_hybrid(movie_id, k=50):
            mid = item['movie_id']
            if mid in all_candidates:
                e = all_candidates[mid]
                e['content_similarity'] = max(e['content_similarity'], item['content_similarity'])
                e['behavioral_similarity'] = max(e['behavioral_similarity'], item['behavioral_similarity'])
            else:
                all_candidates[mid] = item.copy()
    candidates = list(all_candidates.values())
    candidates = filter_watched(candidates, user_id)
    candidates = add_popularity_scores(candidates)
    return rank_three_signals(candidates, content_weight, behavioral_weight, k)

In [ ]:
# Compare all three approaches
collab_recs = recommend_collab('1', k=5)
content_recs = recommend_content_based('1', k=5)
hybrid_recs = recommend_hybrid('1', k=5)

print(f"Recommendations for User 1\n")
COL_W = 32
def _fit(s, w=COL_W):
    s = str(s)
    return s if len(s) <= w else s[: w - 1] + "…"

print(f"{'#':<3} {'Collaborative':<{COL_W}} {'Content-Based':<{COL_W}} {'Hybrid':<{COL_W}}")
print('-' * (3 + 1 + (COL_W + 1) * 3))
for i in range(5):
    ct = movies[movies['movieId'] == collab_recs[i]['movie_id']]['title'].values[0]
    cbt = movies[movies['movieId'] == content_recs[i]['movie_id']]['title'].values[0]
    ht = movies[movies['movieId'] == hybrid_recs[i]['movie_id']]['title'].values[0]
    print(f"{i+1:<3} {_fit(ct):<{COL_W}} {_fit(cbt):<{COL_W}} {_fit(ht):<{COL_W}}")


### Exploring different weight configurations

In [ ]:
configs = [
    ('Content-heavy',    0.6, 0.2),
    ('Balanced',         0.3, 0.5),
    ('Behavioral-heavy', 0.1, 0.7),
]
for name, cw, bw in configs:
    recs = recommend_hybrid('1', k=5, content_weight=cw, behavioral_weight=bw)
    print(f"\n{name} (content={cw}, behavioral={bw}, popularity={1-cw-bw:.1f}):")
    for i, item in enumerate(recs, 1):
        title = movies[movies['movieId'] == item['movie_id']]['title'].values[0]
        print(f"  {i}. {title} (score: {item['final_score']:.3f})")

## Using the `recsys` Framework

The content-based retrieval also lives in the framework as `TFIDFContentRetrieval`. You can swap it into the same `FourStageRecommender` pipeline. 

In [ ]:
from recsys.fourstage_recsys.retrieval.tf_idf_content_retrieval import TFIDFContentRetrieval

content_retrieval = TFIDFContentRetrieval(movies, content_vectors)

# Test it
framework_candidates = content_retrieval.retrieve_similar_by_content('1', k=5)
print("Framework content-based retrieval for Toy Story:")
for item in framework_candidates:
    title = movies[movies['movieId'] == item.item_id]['title'].values[0]
    print(f"  {title}: {item.scores['content_similarity']:.3f}")

The framework retrieval produces the same results as our inline code. You can wire `TFIDFContentRetrieval` into `FourStageRecommender` the same way we wired in `ItemKNNRetrieval` in the first notebook — the pipeline structure doesn't change, only the retrieval implementation does.